In [12]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [13]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [14]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [15]:
# Read image data, feed into Claude
with open("./images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8");

messages = []
add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_011CevuGh9ZmDNSVkcLLRQwr', container=None, content=[TextBlock(citations=None, text='# Satellite Image Fire Risk Analysis\n\n## 1. Residence Identification\nThe primary residence is a single-story structure with a light-colored (gray/beige) roof located in the center of the image, featuring an irregular or L-shaped footprint with what appears to be a driveway or cleared access area on the western side, surrounded by dense forest vegetation on all sides.\n\n## 2. Tree Overhang Analysis\nMultiple large tree canopies directly overhang the residence roof, with dense foliage covering approximately 50-75% of the roof surface, particularly concentrated on the northern, eastern, and southern portions of the structure, while some portions of the western roof section appear more exposed.\n\n## 3. Fire Risk Assessment\nThe overhanging trees create significant wildfire vulnerability by providing direct fuel continuity to the roof structure, establishing multiple ember catch points a

In [16]:
# Read pdf file, feed into Claude
with open("./pdf/earth.pdf", "rb") as f:
    document_bytes = base64.standard_b64encode(f.read()).decode("utf-8");

messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": document_bytes,
        },
        "title": "earth.pdf",
        "citations": {"enabled": True},
    },
    {
        "type": "text",
        "text": "summarize the document in one short sentence",
    }
])

chat(messages)

Message(id='msg_011CevuJ9NBD4Lqv11pzMhiq', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="Earth\r\nThe Blue Marble, Apollo 17, December 1972\r\nDesignations\r\nAlternative\r\nnames\r\nThe world · The globe ·\r\nTerra · Tellus · Gaia ·\r\nMother Earth · Sol III\r\nAdjectives Earthly · Terrestrial · Terran\r\n· Tellurian\r\nSymbol and\r\nOrbital characteristics\r\nEpoch J2000\r\n[n 1]\r\nAphelion 152 097 597 km\r\nPerihelion 147 098 450 km\r\n[n 2]\r\nSemi-major axis 149 598 023 km\r\n[1]\r\nEccentricity 0.016 7086\r\n[1]\r\nOrbital period\r\n(sidereal)\r\n365.256 363 004 d\r\n[2]\r\n(1.000 017 420 96 aj)\r\nAverage orbital\r\nspeed\r\n29.7827 km/s\r\n[3]\r\nMean anomaly 358.617°\r\nInclination 7.155° – Sun's equator;\r\nEarth\r\nEarth is the third planet from the Sun and the only\r\nastronomical object known to harbor life. ", document_index=0, document_title='earth.pdf', end_page_number=2, file_id=None, start_page_number=1, type='page_location')], text='